In [1]:
import pandas as pd

df = pd.read_csv("../datasets/hotel_bookings.csv")
df.shape

(119390, 32)

In [2]:
# Nulos en children (4 filas, ~0.003%)
df["children"] = df["children"].fillna(0)
df["children"] = df["children"].astype(int)

In [3]:
# Nulos en country (488 filas, ~0.4%)
df["country"] = df["country"].fillna("Unknown")

In [4]:
# Nulos en agent (16.340 filas, ~13.7%)
print(df["agent"].min())
df["agent"] = df["agent"].fillna(0)

1.0


In [5]:
# Nulos en company (112.593 filas, ~94.3%)
df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["company"])

## Nota sobre duplicados

El dataset no incluye un identificador único de reserva, por lo que no es posible distinguir con certeza entre duplicados por error de registro y coincidencias legítimas entre reservas distintas con características similares.

Se detectó que, tras eliminar duplicados exactos, la proporción de cancelaciones baja de 37.0% a 27.5%, lo que indica que las reservas canceladas estaban sobrerrepresentadas entre los duplicados. Se decide eliminar los duplicados de todos modos, asumiendo que la mayoría corresponden a errores de registro dado el alto porcentaje (26.8%), pero se documenta esta decisión como una limitación conocida del dataset y del análisis.

In [6]:
# Duplicados (26.8% de las filas)
print("Antes:")
print(df["is_canceled"].value_counts(normalize=True))
print(df.shape)

df = df.drop_duplicates()

print("\nDespués:")
print(df["is_canceled"].value_counts(normalize=True))
print(df.shape)

Antes:
is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64
(119390, 32)

Después:
is_canceled
0    0.725089
1    0.274911
Name: proportion, dtype: float64
(87392, 32)


In [7]:
# Verificación nulos
df.isnull().sum().sum()

np.int64(0)

In [8]:
# Corrección tipo de dato
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])

In [9]:
# Anomalía: reservas sin huéspedes
sin_huespedes = df[(df["adults"] + df["children"] + df["babies"]) == 0]
sin_huespedes.shape[0]

166

In [10]:
df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
df.shape

(87226, 32)

In [11]:
# Guardar el dataset limpio
df.to_csv("../datasets/hotel_bookings_clean.csv", index=False)